In [3]:
# ==========================================
# 🚀 MarketMamba V4.0: 獨立推論引擎 (Cell 1/7) - 環境建置
# ==========================================
import os
from google.colab import drive
drive.mount('/content/drive')
print("🚀 啟動 V4.0 推論引擎環境建置...")

print("🚀 啟動 A100 極速武裝程序 (免授權直連模式)...")
!pip install -q yfinance pandas numpy requests torch-geometric

# 1. 建立暫存資料夾
os.makedirs("mamba_core", exist_ok=True)
os.chdir("mamba_core")

# 2. 從 GitHub Releases 下載 (拿掉 -O 參數，保留原始檔名)
print("📥 正在從 GitHub 下載 Mamba 引擎與 V3.1 大腦...")
!wget -q "https://github.com/FrankChen0930/MarketMamba/releases/download/whl-for-mamba/causal_conv1d-1.6.0-cp312-cp312-linux_x86_64.whl"
!wget -q "https://github.com/FrankChen0930/MarketMamba/releases/download/whl-for-mamba/mamba_ssm-2.3.0-cp312-cp312-linux_x86_64.whl"

# 模型權重檔 .pth 沒有嚴格檔名限制，可以保留 -O 方便我們 Cell 3 讀取
#!wget -q -O MarketMamba_V3_Trajectory.pth "https://github.com/FrankChen0930/MarketMamba/releases/download/v3.1-core/MarketMamba_V3_Trajectory.pth"

# 3. 本地端暴力安裝 (使用萬用字元 * 自動匹配下載下來的那一長串檔名)
print("📦 安裝 Mamba 底層加速套件...")
!pip install -q causal_conv1d-*.whl
!pip install -q mamba_ssm-*.whl

# 切回預設目錄
os.chdir("/content")
print("✅ 環境載入與武器配發完成！MarketMamba V4.0 引擎啟動。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 啟動 V4.0 推論引擎環境建置...
🚀 啟動 A100 極速武裝程序 (免授權直連模式)...
📥 正在從 GitHub 下載 Mamba 引擎與 V3.1 大腦...
📦 安裝 Mamba 底層加速套件...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.5 MB/s eta 0:00:00
✅ 環境載入與武器配發完成！MarketMamba V4.0 引擎啟動。


In [4]:
# ==========================================
# 🧠 MarketMamba V4.0: 獨立推論引擎 (Cell 2/7) - 大腦架構定義
# ==========================================
import torch
import torch.nn as nn
import math
from mamba_ssm import Mamba
import random
import numpy as np

# 👇 新增：工業級亂數種子固定器
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("🌱 亂數種子已固定為 42，確保推論結果具備 100% 絕對重現性！")
print("🧬 正在繪製 V4.0 巨獸大腦藍圖 (Surgery 實盤完全體)...")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# 👇 替換成手術後、沒有 Attention 的版本
class MarketMambaV4_Surgery(nn.Module):
    def __init__(self, input_dim, seq_len=120, d_model=512, pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=seq_len)

        self.mamba_layers = nn.ModuleList()
        self.mamba_norms = nn.ModuleList()
        for _ in range(num_mamba_layers):
            self.mamba_layers.append(Mamba(d_model=d_model, d_state=d_state, d_conv=4, expand=2))
            self.mamba_norms.append(nn.LayerNorm(d_model))

        self.dropout = nn.Dropout(dropout_rate)

        self.trajectory_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_model // 2, pred_days)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        for mamba, norm in zip(self.mamba_layers, self.mamba_norms):
            x = x + self.dropout(mamba(norm(x)))

        x_temporal = x[:, -1, :]
        return self.trajectory_head(x_temporal)

print("✅ 藍圖準備完畢！")

🌱 亂數種子已固定為 42，確保推論結果具備 100% 絕對重現性！
🧬 正在繪製 V4.0 巨獸大腦藍圖 (Surgery 實盤完全體)...
✅ 藍圖準備完畢！


In [5]:
# ==========================================
# 🔮 MarketMamba V4.0: 獨立推論引擎 (Cell 3/7) - 實盤推演
# ==========================================
import os
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

print("🔮 啟動 V4.0 God-Mode 實盤推論...")

ROOT_DIR = '/content/drive/MyDrive/MarketMamba_V4'
data_path = os.path.join(ROOT_DIR, 'Processed_Features', 'Mamba_60D_Matrix.parquet')
# 🚨 修正檔名：讀取最新煉出來的實盤大腦
model_path = os.path.join(ROOT_DIR, 'Models', 'V4_GodMode_Production.pth')
output_csv_path = os.path.join(ROOT_DIR, 'V4_Trajectory_Predictions.csv')

# 1. 載入資料與對齊標準化刻度
df = pd.read_parquet(data_path)
df = df.sort_values(['stock_id', 'Date']).reset_index(drop=True)
exclude_cols = ['stock_id', 'Date', 'Future_5d_Return']
feature_cols = [col for col in df.columns if col not in exclude_cols]

train_mask = df['Date'] < '2025-01-01'
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, feature_cols])

# 2. 擷取最新 120 天快照
SEQ_LEN = 120
latest_date = df['Date'].max()
print(f"📸 擷取最新市場快照 (日期: {latest_date.strftime('%Y-%m-%d')})...")

last_day_stocks = df[df['Date'] == latest_date]['stock_id'].unique()
df_active = df[df['stock_id'].isin(last_day_stocks)].copy()
df_snapshot = df_active.groupby('stock_id').tail(SEQ_LEN)

stock_counts = df_snapshot['stock_id'].value_counts()
valid_stocks = stock_counts[stock_counts == SEQ_LEN].index
df_snapshot = df_snapshot[df_snapshot['stock_id'].isin(valid_stocks)].copy()

active_stock_list = sorted(df_snapshot['stock_id'].unique())
num_stocks = len(active_stock_list)
print(f"  👉 符合 120 天軌跡條件的活躍股票: {num_stocks} 檔")

df_snapshot.loc[:, feature_cols] = scaler.transform(df_snapshot[feature_cols])
X_infer = df_snapshot[feature_cols].values.reshape(num_stocks, SEQ_LEN, len(feature_cols))
X_tensor = torch.tensor(X_infer, dtype=torch.float32)

# 3. 載入模型權重進行預測
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🧠 載入實盤完全體大腦 (裝置: {device})...")

# 🚨 修正為使用手術版模型 (拔掉了 n_heads 參數)
model = MarketMambaV4_Surgery(
    input_dim=len(feature_cols), seq_len=SEQ_LEN, d_model=512,
    pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4
).to(device)

if not os.path.exists(model_path):
    print(f"⚠️ 找不到權重檔: {model_path} (請確認訓練已完成！)")
else:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    print("⚡ 啟動上帝視角，推演未來 30 天軌跡...")
    with torch.no_grad():
        X_tensor = X_tensor.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            predictions = model(X_tensor)

    preds_np = predictions.cpu().numpy()

    # 4. 封裝並輸出
    col_names = ['stock_id', 'Base_Date'] + [f'Day_{i}' for i in range(1, 31)]
    df_results = pd.DataFrame(columns=col_names)
    df_results['stock_id'] = active_stock_list
    df_results['Base_Date'] = latest_date
    for i in range(30):
        df_results[f'Day_{i+1}'] = preds_np[:, i]

    df_results['Expected_30D_Return'] = preds_np[:, -1]
    df_results = df_results.sort_values('Expected_30D_Return', ascending=False).reset_index(drop=True)
    df_results.to_csv(output_csv_path, index=False)

    print(f"\n🎉 預測完成！全市場 30 天軌跡已存檔至: {output_csv_path}")
    display(df_results[['stock_id', 'Expected_30D_Return', 'Day_5', 'Day_10', 'Day_30']].head())

🔮 啟動 V4.0 God-Mode 實盤推論...
📸 擷取最新市場快照 (日期: 2026-02-26)...
  👉 符合 120 天軌跡條件的活躍股票: 2433 檔


/tmp/ipykernel_476/1597319394.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.9488624   6.58016378  4.38882449 ... -0.18076558 -0.17847881
 -0.18297057]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_snapshot.loc[:, feature_cols] = scaler.transform(df_snapshot[feature_cols])
/tmp/ipykernel_476/1597319394.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.54158049 -0.54158049 -0.54158049 ... -0.54158049 -0.54158049
 -0.54158049]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_snapshot.loc[:, feature_cols] = scaler.transform(df_snapshot[feature_cols])


🧠 載入實盤完全體大腦 (裝置: cuda)...
⚡ 啟動上帝視角，推演未來 30 天軌跡...

🎉 預測完成！全市場 30 天軌跡已存檔至: /content/drive/MyDrive/MarketMamba_V4/V4_Trajectory_Predictions.csv


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,stock_id,Expected_30D_Return,Day_5,Day_10,Day_30
0,0050,0.018234,0.003639,0.005116,0.018234
1,6160,0.018234,0.003639,0.005116,0.018234
2,6151,0.018234,0.003639,0.005116,0.018234
3,6152,0.018234,0.003639,0.005116,0.018234
4,6153,0.018234,0.003639,0.005116,0.018234


In [4]:
# ==========================================
# 🩺 巨獸腦部斷層掃描 (檢查 A100 算力是否能被拯救)
# ==========================================
import torch

print("🩺 啟動大腦斷層掃描...")

# 確保模型在推論模式
model.eval()

with torch.no_grad():
    # 我們只抽前 500 檔股票來測試
    test_X = X_tensor[:500].to(device)

    # 1. 檢查輸入特徵是否正常
    print(f"📊 輸入特徵 X 變異數: {test_X.std().item():.4f} (如果接近 0，代表特徵全毀)")

    # 2. 測試 Mamba 獨立思考層
    x = model.embedding(test_X)
    x = model.pos_encoder(x)
    for mamba, norm in zip(model.mamba_layers, model.mamba_norms):
        x = x + model.dropout(mamba(norm(x)))
    x_temporal = x[:, -1, :]
    mamba_std = x_temporal.std(dim=0).mean().item()
    print(f"🧠 Mamba 層輸出變異數: {mamba_std:.4f} (如果大於 0.1，代表 Mamba 活著！A100 沒白跑！)")

    # 3. 測試 上帝視角 Attention 層
    x_graph = x_temporal.unsqueeze(0)
    norm_x = model.attn_norm(x_graph)
    attn_out, _ = model.cross_stock_attn(norm_x, norm_x, norm_x)
    attn_std = attn_out.squeeze(0).std(dim=0).mean().item()
    print(f"👁️ Attention 層輸出變異數: {attn_std:.4f} (如果接近 0，兇手就是它！)")

    # 4. 測試 最終輸出頭
    x_final = x_temporal + attn_out.squeeze(0)
    preds = model.trajectory_head(x_final)
    print(f"📉 最終預測值變異數: {preds.std(dim=0).mean().item():.6f} (就是這個導致大家數字都一樣)")

🩺 啟動大腦斷層掃描...
📊 輸入特徵 X 變異數: 1.6302 (如果接近 0，代表特徵全毀)
🧠 Mamba 層輸出變異數: 0.2135 (如果大於 0.1，代表 Mamba 活著！A100 沒白跑！)


AttributeError: 'MarketMambaV4_Surgery' object has no attribute 'attn_norm'

In [6]:
# ==========================================
# 🌉 MarketMamba V4.0: 獨立推論引擎 (Cell 4/7) - 神經元資料橋接器
# ==========================================
import os
import pandas as pd
import numpy as np

print("🌉 啟動 V4->V3 資料橋接器，準備對接 Streamlit 網站...")

ROOT_DIR = '/content/drive/MyDrive/MarketMamba_V4'
v4_raw_csv = os.path.join(ROOT_DIR, 'V4_Trajectory_Predictions.csv')

# 網站需要的兩個核心檔案
df_traj_path = os.path.join(ROOT_DIR, 'df_traj.csv')
df_kelly_path = os.path.join(ROOT_DIR, 'df_kelly.csv')

# 1. 讀取 V4 巨獸剛剛產生的原始軌跡
if not os.path.exists(v4_raw_csv):
    raise FileNotFoundError("❌ 找不到 V4 軌跡預測檔，請確認 Cell 3 已成功執行！")

df_raw = pd.read_csv(v4_raw_csv)

# ==========================================
# 📁 產出 1: df_traj.csv (純軌跡檔)
# ==========================================
# 網站預期欄位: Ticker, Day_1, Day_2, ..., Day_30
df_traj = df_raw.rename(columns={'stock_id': 'Ticker'})
cols_to_keep = ['Ticker'] + [f'Day_{i}' for i in range(1, 31)]
df_traj = df_traj[cols_to_keep]

df_traj.to_csv(df_traj_path, index=False)
print(f"✅ 已生成網站軌跡檔: df_traj.csv")

# ==========================================
# 📁 產出 2: df_kelly.csv (資金配置與風險指標)
# ==========================================
# 網站預期欄位: Ticker, Volatility_Risk, Sharpe_Score, Suggested_Weight, Exp_Return_15D
df_kelly = pd.DataFrame()
df_kelly['Ticker'] = df_raw['stock_id']
df_kelly['Exp_Return_15D'] = df_raw['Day_15']

# 💡 計算波動率 (Volatility_Risk): 取未來 30 天預測軌跡的標準差作為風險指標
traj_values = df_raw[[f'Day_{i}' for i in range(1, 31)]].values
volatility = np.std(traj_values, axis=1) + 1e-6 # 加上 1e-6 防止除以零
df_kelly['Volatility_Risk'] = volatility

# 💡 計算夏普值 (Sharpe_Score): 預期報酬 / 波動風險
df_kelly['Sharpe_Score'] = df_raw['Expected_30D_Return'] / volatility

# 依夏普值由高到低排序，選出 CP 值最高的股票
df_kelly = df_kelly.sort_values('Sharpe_Score', ascending=False).reset_index(drop=True)

# 💡 計算凱利資金佔比 (Suggested_Weight): 針對前 10 名最強勢股進行權重分配
df_kelly['Suggested_Weight'] = 0.0
top_10_sharpe = df_kelly.loc[:9, 'Sharpe_Score'].values

# 只對正的夏普值分配資金 (安全防護)
valid_sharpe = np.maximum(top_10_sharpe, 0)
if np.sum(valid_sharpe) > 0:
    weights = valid_sharpe / np.sum(valid_sharpe)
    df_kelly.loc[:9, 'Suggested_Weight'] = weights
else:
    # 如果大盤崩跌，全部空手
    df_kelly.loc[:9, 'Suggested_Weight'] = 0.0

df_kelly.to_csv(df_kelly_path, index=False)
print(f"✅ 已生成網站資金盤: df_kelly.csv")

print("\n🎉 橋接完成！現在產出的檔案已經完美符合你 app.py 的格式需求！")

🌉 啟動 V4->V3 資料橋接器，準備對接 Streamlit 網站...
✅ 已生成網站軌跡檔: df_traj.csv
✅ 已生成網站資金盤: df_kelly.csv

🎉 橋接完成！現在產出的檔案已經完美符合你 app.py 的格式需求！


In [7]:
# ==========================================
# 🤖 MarketMamba V4.0: 獨立推論引擎 (Cell 5/7) - 雲端全自動操盤手
# ==========================================
import os
import json
import pandas as pd
import yfinance as yf
from datetime import datetime, timezone, timedelta

print("🤖 召喚雲端全自動操盤手，開始執行每日結算與調倉...")

ROOT_DIR = '/content/drive/MyDrive/MarketMamba_V4'
df_kelly_path = os.path.join(ROOT_DIR, 'df_kelly.csv')
ledger_path = os.path.join(ROOT_DIR, 'robot_ledger.json')

# 1. 讀取剛剛橋接好的凱利資金盤
df_kelly = pd.read_csv(df_kelly_path)

# 2. 讀取或建立實盤帳本
if os.path.exists(ledger_path):
    with open(ledger_path, 'r', encoding='utf-8') as f:
        ledger = json.load(f)
    print("📥 成功讀取雲端歷史帳本！")
else:
    ledger = {
        "start_date": (datetime.now(timezone.utc) + timedelta(hours=8)).strftime("%Y-%m-%d"),
        "cash": 1000000.0,
        "holdings": {},
        "history": []
    }
    print("🆕 建立全新實盤帳本，注入初始資金 100 萬！")

# 3. 超強防呆抓價引擎 (移植自你的 V3.1)
def get_current_prices(tickers):
    prices = {}
    for ticker in tickers:
        clean_ticker = str(ticker).split('.')[0].strip()
        for suffix in ['.TW', '.TWO']:
            try:
                hist = yf.Ticker(f"{clean_ticker}{suffix}").history(period="1d")
                if not hist.empty:
                    prices[str(ticker)] = float(hist['Close'].iloc[-1])
                    break
            except:
                continue
    return prices

# 4. 準備目標權重與標的
top_10 = df_kelly.head(10)
target_tickers = top_10['Ticker'].astype(str).tolist()
target_weights = top_10['Suggested_Weight'].values

# 需要抓取「現有庫存」+「目標名單」的最新股價
all_needed_tickers = list(set(list(ledger["holdings"].keys()) + target_tickers))
print("🔄 正在連線 Yahoo Finance 獲取最新報價...")
live_prices = get_current_prices(all_needed_tickers)

# ==========================================
# ⚙️ 執行調倉 (Rebalancing)
# ==========================================
# (1) 賣出不在凱利前 10 名的股票
for t in list(ledger["holdings"].keys()):
    if t not in target_tickers:
        shares_to_sell = ledger["holdings"][t]["shares"]
        sell_price = live_prices.get(t, ledger["holdings"][t]["cost"])
        ledger["cash"] += shares_to_sell * sell_price
        del ledger["holdings"][t]
        print(f"  🔴 賣出剔除名單: {t} (套現: {shares_to_sell * sell_price:.0f})")

# (2) 計算當下總淨值
current_total_funds = ledger["cash"]
for t, data in ledger["holdings"].items():
    current_total_funds += data["shares"] * live_prices.get(t, data["cost"])

# (3) 依凱利權重買進/加碼
for ticker, weight in zip(target_tickers, target_weights):
    if ticker not in live_prices or weight <= 0: continue

    target_value = current_total_funds * weight
    current_price = live_prices[ticker]
    target_shares = int(target_value // current_price)
    current_shares = ledger["holdings"].get(ticker, {}).get("shares", 0)
    shares_to_buy = target_shares - current_shares

    if shares_to_buy > 0 and ledger["cash"] >= (shares_to_buy * current_price):
        ledger["cash"] -= (shares_to_buy * current_price)
        if ticker in ledger["holdings"]:
            old_cost = ledger["holdings"][ticker]["cost"]
            old_shares = ledger["holdings"][ticker]["shares"]
            new_avg_cost = ((old_cost * old_shares) + (current_price * shares_to_buy)) / target_shares
            ledger["holdings"][ticker] = {"shares": target_shares, "cost": new_avg_cost}
        else:
            ledger["holdings"][ticker] = {"shares": shares_to_buy, "cost": current_price}
        print(f"  🟢 買進建倉: {ticker} (數量: {shares_to_buy} 股, 價格: {current_price})")

# 5. 更新淨值歷史曲線並存檔
today_str = (datetime.now(timezone.utc) + timedelta(hours=8)).strftime("%Y-%m-%d %H:%M")
# 避免同一天重複紀錄
if len(ledger["history"]) > 0 and ledger["history"][-1]["date"][:10] == today_str[:10]:
    ledger["history"][-1] = {"date": today_str, "equity": current_total_funds}
else:
    ledger["history"].append({"date": today_str, "equity": current_total_funds})

with open(ledger_path, 'w', encoding='utf-8') as f:
    json.dump(ledger, f, indent=4)

print(f"✅ 今日調倉與帳本結算完畢！最新總淨值: {current_total_funds:,.0f} TWD")

🤖 召喚雲端全自動操盤手，開始執行每日結算與調倉...
📥 成功讀取雲端歷史帳本！
🔄 正在連線 Yahoo Finance 獲取最新報價...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 9962.TW"}}}
ERROR:yfinance:$9962.TW: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 2729.TW"}}}
ERROR:yfinance:$2729.TW: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$2640.TW: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$2732.TW: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


✅ 今日調倉與帳本結算完畢！最新總淨值: 988,877 TWD


In [8]:
# ==========================================
# 🚀 MarketMamba V4.0: 自動推論引擎 (Cell 6/7) - GitHub 發布管線
# ==========================================
import os
import datetime
from google.colab import userdata

print("🚀 開始執行自動化發布管線...")

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print("❌ 找不到 GITHUB_TOKEN，請確認左側鑰匙圈有設定正確！")
    raise e

# ⚠️ 注意：執行前請確認這裡的帳號與信箱是否正確！
GITHUB_USER = "FrankChen0930"
GITHUB_EMAIL = "a0966469964@gmail.com"
REPO_NAME = "MarketMamba"

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
ROOT_DIR = '/content/drive/MyDrive/MarketMamba_V4'

# 1. 下載最新 Repo
os.system("rm -rf repo_folder")
os.system(f"git clone {REPO_URL} repo_folder")

# 2. 🚨 修正點：從 Google Drive 把這「三個」核心檔案複製到準備推播的資料夾
os.system(f"cp '{ROOT_DIR}/df_kelly.csv' repo_folder/")
os.system(f"cp '{ROOT_DIR}/df_traj.csv' repo_folder/")
os.system(f"cp '{ROOT_DIR}/robot_ledger.json' repo_folder/")

# 3. 進入資料夾準備推播
os.chdir("repo_folder")

os.system(f"git config user.name '{GITHUB_USER}'")
os.system(f"git config user.email '{GITHUB_EMAIL}'")

# 4. 更新時間戳記
tw_time = (datetime.datetime.utcnow() + datetime.timedelta(hours=8)).strftime("%Y-%m-%d %H:%M:%S")
with open("update_time.txt", "w") as f:
    f.write(tw_time)

# 5. 將所有更新的檔案加入 Git 追蹤
os.system("git add df_kelly.csv df_traj.csv update_time.txt robot_ledger.json")

# 6. 提交並 Push (加上 V4.0 標籤)
os.system(f"git commit -m '🤖 Auto-update: V4.0 Daily Inference & Ledger ({tw_time})'")
push_result = os.system("git push origin main")

if push_result == 0:
    print(f"🎉 大功告成！今日({tw_time}) 最新的 V4 預測與實盤帳本已成功推送到 GitHub！")
    print("🌐 你的 Streamlit 網站將在幾秒鐘內自動讀取到最新數據！")
else:
    print("⚠️ 推送失敗，請檢查 Token 權限或檔案路徑。")

# 7. 退回主目錄，準備執行下一個關機 Cell
os.chdir("/content")

🚀 開始執行自動化發布管線...


/tmp/ipykernel_476/32277267.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  tw_time = (datetime.datetime.utcnow() + datetime.timedelta(hours=8)).strftime("%Y-%m-%d %H:%M:%S")


🎉 大功告成！今日(2026-03-12 23:27:06) 最新的 V4 預測與實盤帳本已成功推送到 GitHub！
🌐 你的 Streamlit 網站將在幾秒鐘內自動讀取到最新數據！


In [9]:
# ==========================================
# 🛑 MarketMamba V4.0: 獨立推論引擎 (Cell 7/7) - 自動關機程序
# ==========================================
from google.colab import runtime
import time

print("💤 今日 V4.0 推論已全數完成。")
print("🔌 為了節省算力點數，系統將在 5 秒後自動切斷執行階段電源...")
time.sleep(5)

# 執行自爆指令，強制回收虛擬機與 GPU
runtime.unassign()

💤 今日 V4.0 推論已全數完成。
🔌 為了節省算力點數，系統將在 5 秒後自動切斷執行階段電源...
